In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

In [ ]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

In [ ]:
Groq_API_KEY=""
os.environ["Groq_API_KEY"]=Groq_API_KEY

groq_client=Groq(api_key=Groq_API_KEY)
print("Groq API client initialized")

Groq API client initialized


In [ ]:
df=pd.read_csv('/content/college_notes.csv')

print("shape of data set: ",df.shape)
print("\n Column names :",df.columns.tolist())

print(df.head(3))

shape of data set:  (15, 4)

 Column names : ['note_id', 'subject', 'topic', 'content']
   note_id           subject             topic  \
0        1  Data Engineering     ETL Pipelines   
1        2  Data Engineering  Data Warehousing   
2        3  Data Engineering      Apache Spark   

                                             content  
0  ETL stands for Extract, Transform, Load. It is...  
1  A data warehouse is a central repository that ...  
2  Apache Spark is an open-source distributed com...  


In [ ]:
print("subject in dataset:")
print(df['subject'].value_counts())
print("\n Sample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\n Length of content (number of character) for each notes")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].head(3).to_string(index=False))

subject in dataset:
subject
Data Engineering    5
GenAI               5
Machine Learning    3
Python              2
Name: count, dtype: int64

 Sample of topics:
 note_id          subject                  topic
       1 Data Engineering          ETL Pipelines
       2 Data Engineering       Data Warehousing
       3 Data Engineering           Apache Spark
       4 Data Engineering Medallion Architecture
       5 Data Engineering         Data Pipelines
       6 Machine Learning      Linear Regression
       7 Machine Learning    Feature Engineering
       8 Machine Learning       Model Evaluation
       9            GenAI  Large Language Models
      10            GenAI     Prompt Engineering
      11            GenAI             Embeddings
      12            GenAI       Vector Databases
      13            GenAI            RAG Systems
      14           Python         Pandas Library
      15           Python        API Integration

 Length of content (number of character) for each not

In [ ]:
from chromadb.api.types import Metadata
documents=df['content'].tolist()

ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas=[
    {"subject ": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]


print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID: {ids[0]}")
print(f"First metadata: {metadatas[0]}")
print(f"First 100 chars of doc:{documents[0][:100]}....")

Total chunks prepared: 15
First document ID: note_1
First metadata: {'subject ': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc:ETL stands for Extract, Transform, Load. It is a process used in data engineering to move data from ....


In [ ]:
#embedding model

embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print("embedding successfully loaded")
test_embedding=embedding_model.encode("This is a test sentence")
print(f"Test embedding shape:{test_embedding.shape}")

print(f"First 5 valuesof test embedding: {test_embedding[:5]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding successfully loaded
Test embedding shape:(384,)
First 5 valuesof test embedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [ ]:
chroma_client=chromadb.Client()

collection=chroma_client.get_or_create_collection(name="college_notes_rag")
print("ChromaDB client created.")
print(f"collection name: college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

ChromaDB client created.
collection name: college_notes_rag
Documents in collection so far:0


In [ ]:
print("generating embeddings for all 15 notes...")
print("this may take 15-30 seconds")
embeddings=embedding_model.encode(documents,show_progress_bar=True)
print(f"\n Embedding matrix shape:{embeddings.shape}")
embeddings_list=embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    metadatas=metadatas,
    ids=ids
)

print(f"\n Documents successfully addded to chromaDB")
print(f"Total documents in collection: {collection.count()}")

generating embeddings for all 15 notes...
this may take 15-30 seconds


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Embedding matrix shape:(15, 384)

 Documents successfully addded to chromaDB
Total documents in collection: 15


In [32]:
def retrive_revalent_chunks(question,top_k=3):
  question_embedding=embedding_model.encode(question).tolist()


  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results

print("Retrival function defined successfully..")
print("Function: retrival_revalent_chunks(question,top_k=3)")

Retrival function defined successfully..
Function: retrival_revalent_chunks(question,top_k=3)


In [39]:
test_question="which Requires learning specific coding syntax.?"
print(f"Test Question: {test_question}")
print("="*60)

results=retrive_revalent_chunks(test_question,top_k=3)

print("\nTop 3 retrived chunks:")
print("="*60)

for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
     print(f"\n Results{i+1}:")
     print(f"   Subject  :{meta['subject ']}")
     print(f"   Topics   :{meta['topic']}")
     print(f"   Distance : {dist:.4f}")

     print(f"  Content  : {doc[:120]}....")

Test Question: which Requires learning specific coding syntax.?

Top 3 retrived chunks:

 Results1:
   Subject  :GenAI
   Topics   :Prompt Engineering
   Distance : 1.3833
  Content  : Prompt engineering is the practice of designing and refining the instructions given to a large language model to get the....

 Results2:
   Subject  :GenAI
   Topics   :Large Language Models
   Distance : 1.4247
  Content  : A Large Language Model or LLM is a type of artificial intelligence trained on massive amounts of text data to understand....

 Results3:
   Subject  :Machine Learning
   Topics   :Feature Engineering
   Distance : 1.4329
  Content  : Feature engineering is the process of selecting, creating, and transforming raw data variables called features to improv....


In [14]:
def build_context_from_results(results):
  context_parts=[]

  for i,(doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):

      chunk_text=f"[Source {i+1}: {meta['subject ']}-{meta['topic']}]\n{doc}"
      print("\n Top 3 Retrived chunks:")

In [13]:
def generate_rag_answer(question, context):

    system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer only using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer the question."
3. Do not use your general training knowledge.
4. Keep the answer clear and concise.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )

    answer = response.choices[0].message.content

    return answer


print("RAG generation function defined successfully...")
print("Function: generate_rag_answer(question, context)")

RAG generation function defined successfully...
Function: generate_rag_answer(question, context)


In [15]:
def ask_college_assistant(question, top_k=3, verbose=True):

    if verbose:
        print(f"\nQuestion: {question}")
        print("=" * 60)

    # Retrieve relevant chunks
    results = retrive_revalent_chunks(question, top_k=top_k)

    if verbose:
        print(f"Retrieved {top_k} chunks from knowledge base:\n")

        for i, chunk in enumerate(results, start=1):
            print(f"Chunk {i}:")
            print(chunk)
            print("-" * 60)

    # Combine chunks into a single context
    context = "\n\n".join(results)

    if verbose:
        print("\nGenerating answer...")
        print("=" * 60)

    # Generate RAG answer
    answer = generate_rag_answer(question, context)

    if verbose:
        print("\nAnswer:")
        print(answer)

    return answer